In [1]:
# Optional ipykernel package installation
#!pip install ipykernel
# Once done, don't forget to install venv as described in README.md
# And choose this venv for runnning your jupyter kernel

In [2]:
# Imports
import sys
import os
import io
sys.path.append('..')

# Simulate ETL process

---
Extract from datalake:  
read dataset from S3  
import as pandas dataframe

In [3]:
# Setup Logging 
import logging
# Configure root logger
logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] %(name)s: %(message)s'
)
logging.getLogger("pipeline.libs.src.aws").setLevel(logging.INFO)
logger = logging.getLogger(__name__)

In [4]:
# Serialize to S3 to make datasets available for further processing
from pipeline.libs.src.utils import get_project_name
from pipeline.libs.src.aws import serialize_to_s3
from pipeline.libs.src.aws import get_serialized_from_S3

import pandas as pd


# Read dataset from S3 (assume existing objet)
project_root_path = get_project_name()  # $PROJECT_NAME to be set in .env
logger.info(f"project_root_path: {project_root_path}")

# Serialize datasets
l_path_to_data = ['../data/raw/df_rentals.pkl', '../data/raw/df_cars.pkl']  # files in data relative to /notebooks
l_df_deserialized = []

logger.info(f"Current working directory: {os.getcwd()}")

for path_to_data in l_path_to_data:
    logger.info(f"Dataset to serialize: {path_to_data}")
    
    # Import dataset from local dir
    df = pd.read_pickle(path_to_data)
    
    path_to_data = path_to_data.replace("../", "").lstrip("/")
    s3_key = f'{project_root_path}/{path_to_data}'
    logger.info(f"s3_key: {s3_key}")
    
    serialize_to_s3(df, s3_key)
    
    # test serialization
    obj = get_serialized_from_S3(s3_key)
    l_df_deserialized.append(obj)

[INFO] __main__: project_root_path: ml-getaround-docker-deploy
[INFO] __main__: Current working directory: /home/fabien/VSCode/dsfs-ft-32/Projets/ml-getaround-docker-deploy/notebooks
[INFO] __main__: Dataset to serialize: ../data/raw/df_rentals.pkl
[INFO] __main__: s3_key: ml-getaround-docker-deploy/data/raw/df_rentals.pkl
[INFO] pipeline.libs.src.aws: Serializing object to S3: s3://jedha-projects/ml-getaround-docker-deploy/data/raw/df_rentals.pkl
[INFO] pipeline.libs.src.aws: Type of Object ready for serializing to S3: <class 'pandas.core.frame.DataFrame'>
[INFO] pipeline.libs.src.aws: Object successfully serialized to S3: s3://jedha-projects/ml-getaround-docker-deploy/data/raw/df_rentals.pkl
[INFO] pipeline.libs.src.aws: Verification successful - The file exists in S3
[INFO] pipeline.libs.src.aws: Deserializing the file from S3: s3://jedha-projects/ml-getaround-docker-deploy/data/raw/df_rentals.pkl
[INFO] pipeline.libs.src.aws: File successfully loaded from S3: s3://jedha-projects/ml

---
# Transform data

Dummy transform operations  
Rename col, create col

In [5]:
# Imports
from pipeline.libs.src.utils import get_project_name
from pipeline.libs.src.aws import get_serialized_from_S3

# Read datasets
project_root_path = get_project_name()  # $PROJECT_NAME to be set in .env
logger.info(f"project_root_path: {project_root_path}")

# Deserialize datasets
l_path_to_data = ['data/raw/df_rentals.pkl', 'data/raw/df_cars.pkl']  # datasets in bucket
l_df_deserialized = []

for path_to_data in l_path_to_data:
    logger.info(f"Dataset to deserialize: {path_to_data}")
    
    s3_key = f'{project_root_path}/{path_to_data}'
    logger.info(f"s3_key: {s3_key}")
    
    df = get_serialized_from_S3(s3_key)
    l_df_deserialized.append(df)

logger.info(f"List of deserialized datasets: {len(l_df_deserialized)} datasets")
for i, df in enumerate(l_df_deserialized):
    logger.info(f"Object {i} deserialized: {type(df)} {df.shape}")

[INFO] __main__: project_root_path: ml-getaround-docker-deploy
[INFO] __main__: Dataset to deserialize: data/raw/df_rentals.pkl
[INFO] __main__: s3_key: ml-getaround-docker-deploy/data/raw/df_rentals.pkl
[INFO] pipeline.libs.src.aws: Deserializing the file from S3: s3://jedha-projects/ml-getaround-docker-deploy/data/raw/df_rentals.pkl
[INFO] pipeline.libs.src.aws: File successfully loaded from S3: s3://jedha-projects/ml-getaround-docker-deploy/data/raw/df_rentals.pkl
[INFO] pipeline.libs.src.aws: Type of Object deserialized from S3: <class 'pandas.core.frame.DataFrame'>
[INFO] pipeline.libs.src.aws: Object successfully deserialized from S3.
[INFO] __main__: Dataset to deserialize: data/raw/df_cars.pkl
[INFO] __main__: s3_key: ml-getaround-docker-deploy/data/raw/df_cars.pkl
[INFO] pipeline.libs.src.aws: Deserializing the file from S3: s3://jedha-projects/ml-getaround-docker-deploy/data/raw/df_cars.pkl
[INFO] pipeline.libs.src.aws: File successfully loaded from S3: s3://jedha-projects/ml

In [6]:
df_cars = l_df_deserialized[1].copy()
df_cars.shape

(4784, 14)

In [7]:
df_cars.head(5)

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183
5,Citroën,152352,225,petrol,black,convertible,True,True,False,False,True,True,True,131


Outliers ont été supprimés dans l'EDA

In [8]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Display proportion of each category in every categorical column
# CATEGORICAL COLUMNS
# Get list of categorical columns
l_categorical_cols = df_cars.select_dtypes(include=["object", "category", "bool"]).columns

# Number of lines needed
rows = (len(l_categorical_cols) + 1) // 2

fig = make_subplots(
    rows=rows,
    cols=2,
    subplot_titles=l_categorical_cols,
    specs=[[{"type": "domain"}, {"type": "domain"}] for _ in range(rows)]
)

# Display proportion of each category in every categorical column
for i, col in enumerate(l_categorical_cols):
    counts = df_cars[col].value_counts().reset_index()
    counts.columns = [col, "count"]
    pie = go.Pie(labels=counts[col], values=counts["count"], name=col, textinfo="percent+label")
    fig.add_trace(pie, row=i // 2 + 1, col=i % 2 + 1)

fig.update_layout(
    title_text="Categorical columns distribution",
    height=440 * rows,
    width=800,
    showlegend=False,
    margin=dict(l=10, r=10, t=10, b=10)  # left, right, top, bottom
)
fig.show()

# NUMERIC COLUMNS
l_numeric_cols = df_cars.select_dtypes(include=["int64", "float64"]).columns
rows = (len(l_numeric_cols) + 1) // 2

fig = make_subplots(
    rows=rows,
    cols=2,
    subplot_titles=l_numeric_cols
)

for i, col in enumerate(l_numeric_cols):
    hist = go.Histogram(x=df_cars[col], name=col)
    fig.add_trace(hist, row=i // 2 + 1, col=i % 2 + 1)

fig.update_layout(
    title_text="Numeric columns distribution",
    height=250 * rows,
    width=800,
    showlegend=False,
    margin=dict(l=10, r=10, t=40, b=10)  # left, right, top, bottom
)

fig.show()

Les colonnes "fuel", "model_key" et "paint_color" ont une <u>cardinalité trop importante</u> qui posera problème lors de l'encodage catégoriel.
On baisse la granularité des valeurs dans ces colonnes en remplaçant soit par une valeur "rare" soit par NaN. On choisit de remplacer les valeurs qui sont présentes moins de 10 fois et on les supprime

In [9]:
# Replace by NaN
df_cars_wo_rares = df_cars.copy()
for col in ["fuel", 'model_key', 'paint_color']:
    df_counts = pd.DataFrame(df_cars_wo_rares[col].value_counts())
    for category in df_counts.loc[df_counts['count']<=10,:].index.to_list():
        for row in df_cars_wo_rares.loc[df_cars_wo_rares[col]==category].index.to_list():
            df_cars_wo_rares.drop(row, axis=0, inplace=True)

print(df_cars.shape)
print(df_cars_wo_rares.shape)

(4784, 14)
(4747, 14)


In [10]:
# Serialize as transformed dataframe .pkl in data/transformed
from pipeline.libs.src.aws import serialize_to_s3

project_root_path = get_project_name()
logger.info(f"project_root_path: {project_root_path}")
path_to_transformed_data = f'data/transformed/df_cars_transformed.pkl'
logger.info(f"path_to_transformed_data: {path_to_transformed_data}")

s3_key = f'{project_root_path}/{path_to_transformed_data}'
logger.info(f"s3_key: {s3_key}")

serialize_to_s3(df_cars_wo_rares, s3_key)

[INFO] __main__: project_root_path: ml-getaround-docker-deploy
[INFO] __main__: path_to_transformed_data: data/transformed/df_cars_transformed.pkl
[INFO] __main__: s3_key: ml-getaround-docker-deploy/data/transformed/df_cars_transformed.pkl
[INFO] pipeline.libs.src.aws: Serializing object to S3: s3://jedha-projects/ml-getaround-docker-deploy/data/transformed/df_cars_transformed.pkl
[INFO] pipeline.libs.src.aws: Type of Object ready for serializing to S3: <class 'pandas.core.frame.DataFrame'>
[INFO] pipeline.libs.src.aws: Object successfully serialized to S3: s3://jedha-projects/ml-getaround-docker-deploy/data/transformed/df_cars_transformed.pkl
[INFO] pipeline.libs.src.aws: Verification successful - The file exists in S3


True

---
# Load data

In [11]:
# Setup Logging
import logging
# Configure root logger
logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] %(name)s: %(message)s'
)
logging.getLogger("pipeline.libs.src.aws").setLevel(logging.INFO)
logger = logging.getLogger(__name__)

In [12]:
# Deserialize from data/transformed
project_root_path = get_project_name()
logger.info(f"project_root_path: {project_root_path}")
path_to_transformed_data = f'data/transformed/df_cars_transformed.pkl'
logger.info(f"path_to_transformed_data: {path_to_transformed_data}")

# Deserialize from Bucket/Project/...
s3_key = f'{project_root_path}/{path_to_transformed_data}'
logger.info(f"s3_key: {s3_key}")

# Load from S3: S3_key = file path relative to bucket name defined in ENV
from pipeline.libs.src.aws import get_serialized_from_S3
dataset = get_serialized_from_S3(s3_key)

[INFO] __main__: project_root_path: ml-getaround-docker-deploy
[INFO] __main__: path_to_transformed_data: data/transformed/df_cars_transformed.pkl
[INFO] __main__: s3_key: ml-getaround-docker-deploy/data/transformed/df_cars_transformed.pkl
[INFO] pipeline.libs.src.aws: Deserializing the file from S3: s3://jedha-projects/ml-getaround-docker-deploy/data/transformed/df_cars_transformed.pkl
[INFO] pipeline.libs.src.aws: File successfully loaded from S3: s3://jedha-projects/ml-getaround-docker-deploy/data/transformed/df_cars_transformed.pkl
[INFO] pipeline.libs.src.aws: Type of Object deserialized from S3: <class 'pandas.core.frame.DataFrame'>
[INFO] pipeline.libs.src.aws: Object successfully deserialized from S3.


---
Sandbox for data manipulation

In [13]:
dataset

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183
5,Citroën,152352,225,petrol,black,convertible,True,True,False,False,True,True,True,131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4838,Toyota,39743,110,diesel,black,van,False,True,False,False,False,False,True,121
4839,Toyota,49832,100,diesel,grey,van,False,True,False,False,False,False,True,132
4840,Toyota,19633,110,diesel,grey,van,False,True,False,False,False,False,True,130
4841,Toyota,27920,110,diesel,brown,van,True,True,False,False,False,False,True,151
